# Preparação do recorte municipal

Este notebook transforma o `recorte` bruto do município em uma camada `preparado`, com campos derivados, datas normalizadas e colunas úteis para a geocodificação.

## Saídas principais

- `estabelecimentos_preparados.parquet`
- `empresas_preparadas.parquet`
- `simples_preparados.parquet`
- `qualidade_preparacao.json`

Saídas auxiliares:

- `cnaes_recorte.parquet`
- `motivos_situacao_recorte.parquet`

In [ ]:
from pathlib import Path
import json
import re

try:
    import pandas as pd
    import pyarrow  # noqa: F401
except ImportError:
    !pip -q install pandas pyarrow
    import pandas as pd
    import pyarrow  # noqa: F401

SNAPSHOT_MES = '2026-04'
SNAPSHOT = Path('/content/dados_brutos/cnpj') / SNAPSHOT_MES
ENTRADA_RECORTE = SNAPSHOT / 'processado' / 'recorte'
SAIDA_PREPARADO = SNAPSHOT / 'processado' / 'preparado'
DOMINIOS = SNAPSHOT / 'extraido' / 'dominios'

SAIDA_PREPARADO.mkdir(parents=True, exist_ok=True)

ARQUIVOS_RECORTE = {
    'estabelecimentos': ENTRADA_RECORTE / 'estabelecimentos_municipio.parquet',
    'empresas': ENTRADA_RECORTE / 'empresas_municipio.parquet',
    'simples': ENTRADA_RECORTE / 'simples_municipio.parquet',
    'cnpjs': ENTRADA_RECORTE / 'cnpjs_basicos_municipio.parquet',
}

for nome, caminho in ARQUIVOS_RECORTE.items():
    if not caminho.exists():
        raise FileNotFoundError(f'Arquivo obrigatório ausente em recorte: {nome} -> {caminho}')

ARQUIVOS_RECORTE

In [ ]:
SITUACOES = {
    '01': 'nula',
    '02': 'ativa',
    '03': 'suspensa',
    '04': 'inapta',
    '08': 'baixada',
}

def normalizar_data_yyyymmdd(serie: pd.Series) -> pd.Series:
    serie_limpa = serie.fillna('').astype(str).str.strip()
    serie_limpa = serie_limpa.replace({'': None, '0': None, '00000000': None})
    return pd.to_datetime(serie_limpa, format='%Y%m%d', errors='coerce')

def limpar_texto(valor: object) -> str:
    if valor is None:
        return ''
    texto = str(valor).strip()
    return re.sub(r'\s+', ' ', texto)

def limpar_numero(valor: object) -> str:
    texto = limpar_texto(valor)
    return re.sub(r'\D', '', texto)

def montar_cnpj_completo(df: pd.DataFrame) -> pd.Series:
    return (
        df['cnpj_basico'].fillna('').astype(str).str.zfill(8)
        + df['cnpj_ordem'].fillna('').astype(str).str.zfill(4)
        + df['cnpj_dv'].fillna('').astype(str).str.zfill(2)
    )

def montar_endereco_texto(df: pd.DataFrame) -> pd.Series:
    partes = [
        (df['tipo_logradouro'].map(limpar_texto) + ' ' + df['logradouro'].map(limpar_texto)).str.strip(),
        df['numero'].map(limpar_texto),
        df['complemento'].map(limpar_texto),
        df['bairro'].map(limpar_texto),
        df['cep'].map(limpar_texto),
        df['uf'].map(limpar_texto),
    ]
    endereco = pd.Series('', index=df.index, dtype='object')
    for parte in partes:
        endereco = (endereco + ' ' + parte.fillna('')).str.strip()
    return endereco.str.replace(r'\s+', ' ', regex=True).str.strip()

def extrair_dominio_por_sufixo(sufixo: str) -> Path | None:
    candidatos = sorted(DOMINIOS.glob(f'*.{sufixo}'))
    return candidatos[0] if candidatos else None


In [ ]:
estabelecimentos = pd.read_parquet(ARQUIVOS_RECORTE['estabelecimentos'])
empresas = pd.read_parquet(ARQUIVOS_RECORTE['empresas'])
simples = pd.read_parquet(ARQUIVOS_RECORTE['simples'])

estabelecimentos.shape, empresas.shape, simples.shape

In [ ]:
estabelecimentos_preparados = estabelecimentos.copy()
estabelecimentos_preparados['snapshot_mes'] = SNAPSHOT_MES
estabelecimentos_preparados['cnpj_completo'] = montar_cnpj_completo(estabelecimentos_preparados)
estabelecimentos_preparados['data_inicio_atividade'] = normalizar_data_yyyymmdd(estabelecimentos_preparados['data_inicio_atividade'])
estabelecimentos_preparados['data_situacao_cadastral'] = normalizar_data_yyyymmdd(estabelecimentos_preparados['data_situacao_cadastral'])
estabelecimentos_preparados['data_situacao_especial'] = normalizar_data_yyyymmdd(estabelecimentos_preparados['data_situacao_especial'])
estabelecimentos_preparados['situacao_cadastral_codigo'] = estabelecimentos_preparados['situacao_cadastral'].fillna('').astype(str).str.zfill(2)
estabelecimentos_preparados['situacao_cadastral_label'] = estabelecimentos_preparados['situacao_cadastral_codigo'].map(SITUACOES).fillna('desconhecida')
estabelecimentos_preparados['flag_ativa'] = estabelecimentos_preparados['situacao_cadastral_codigo'] == '02'
estabelecimentos_preparados['flag_suspensa'] = estabelecimentos_preparados['situacao_cadastral_codigo'] == '03'
estabelecimentos_preparados['flag_inapta'] = estabelecimentos_preparados['situacao_cadastral_codigo'] == '04'
estabelecimentos_preparados['flag_baixada'] = estabelecimentos_preparados['situacao_cadastral_codigo'] == '08'
estabelecimentos_preparados['cep_limpo'] = estabelecimentos_preparados['cep'].map(limpar_numero).str.zfill(8)
estabelecimentos_preparados['bairro_limpo'] = estabelecimentos_preparados['bairro'].map(limpar_texto)
estabelecimentos_preparados['logradouro_limpo'] = estabelecimentos_preparados['logradouro'].map(limpar_texto)
estabelecimentos_preparados['numero_limpo'] = estabelecimentos_preparados['numero'].map(limpar_numero)
estabelecimentos_preparados['endereco_texto'] = montar_endereco_texto(estabelecimentos_preparados)
estabelecimentos_preparados['chave_endereco'] = (
    estabelecimentos_preparados['endereco_texto'].fillna('').str.upper()
    + ' | ' + estabelecimentos_preparados['municipio'].fillna('').astype(str)
    + ' | ' + estabelecimentos_preparados['uf'].fillna('').astype(str)
)
estabelecimentos_preparados['metodo_geocodificacao'] = 'nao_geocodificado'
estabelecimentos_preparados['confianca_geografica'] = 'baixa'
estabelecimentos_preparados['precisa_revisao_geocodificacao'] = True
estabelecimentos_preparados['latitude'] = pd.NA
estabelecimentos_preparados['longitude'] = pd.NA
estabelecimentos_preparados['trilha_processamento'] = '03_preparacao_recorte'

estabelecimentos_preparados[['cnpj_completo', 'situacao_cadastral_label', 'endereco_texto']].head()

In [ ]:
empresas_preparadas = empresas.copy()
empresas_preparadas['snapshot_mes'] = SNAPSHOT_MES
empresas_preparadas['capital_social_numero'] = (
    empresas_preparadas['capital_social']
    .fillna('0')
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)
empresas_preparadas['capital_social_numero'] = pd.to_numeric(empresas_preparadas['capital_social_numero'], errors='coerce')

simples_preparados = simples.copy()
simples_preparados['snapshot_mes'] = SNAPSHOT_MES
for coluna in ['data_opcao_pelo_simples', 'data_exclusao_do_simples', 'data_opcao_pelo_mei', 'data_exclusao_do_mei']:
    simples_preparados[coluna] = normalizar_data_yyyymmdd(simples_preparados[coluna])
simples_preparados['flag_simples_ativo'] = simples_preparados['opcao_pelo_simples'].fillna('').astype(str).str.upper() == 'S'
simples_preparados['flag_mei_ativo'] = simples_preparados['opcao_pelo_mei'].fillna('').astype(str).str.upper() == 'S'

empresas_preparadas.head(2), simples_preparados.head(2)

In [ ]:
cnaes_path = extrair_dominio_por_sufixo('CNAECSV')
motivos_path = extrair_dominio_por_sufixo('MOTICSV')

if cnaes_path is not None:
    cnaes_df = pd.read_csv(cnaes_path, sep=';', header=None, names=['cnae', 'descricao'], dtype=str, encoding='latin1')
    codigos_cnae = set(estabelecimentos_preparados['cnae_fiscal_principal'].dropna().astype(str))
    cnaes_recorte = cnaes_df.loc[cnaes_df['cnae'].astype(str).isin(codigos_cnae)].copy()
    cnaes_recorte.to_parquet(SAIDA_PREPARADO / 'cnaes_recorte.parquet', index=False)

if motivos_path is not None:
    motivos_df = pd.read_csv(motivos_path, sep=';', header=None, names=['motivo', 'descricao'], dtype=str, encoding='latin1')
    motivos_df['motivo'] = motivos_df['motivo'].astype(str).str.zfill(2)
    codigos_motivo = set(estabelecimentos_preparados['motivo_situacao_cadastral'].dropna().astype(str).str.zfill(2))
    motivos_recorte = motivos_df.loc[motivos_df['motivo'].isin(codigos_motivo)].copy()
    motivos_recorte.to_parquet(SAIDA_PREPARADO / 'motivos_situacao_recorte.parquet', index=False)

cnaes_path, motivos_path

In [ ]:
estabelecimentos_preparados.to_parquet(SAIDA_PREPARADO / 'estabelecimentos_preparados.parquet', index=False)
empresas_preparadas.to_parquet(SAIDA_PREPARADO / 'empresas_preparadas.parquet', index=False)
simples_preparados.to_parquet(SAIDA_PREPARADO / 'simples_preparados.parquet', index=False)

qualidade_preparacao = {
    'snapshot_referencia': SNAPSHOT_MES,
    'cidade': 'Praia Grande - SP',
    'quantidade_estabelecimentos': int(len(estabelecimentos_preparados)),
    'quantidade_empresas': int(len(empresas_preparadas)),
    'quantidade_simples': int(len(simples_preparados)),
    'ativos': int(estabelecimentos_preparados['flag_ativa'].sum()),
    'baixadas': int(estabelecimentos_preparados['flag_baixada'].sum()),
    'suspensas': int(estabelecimentos_preparados['flag_suspensa'].sum()),
    'inaptas': int(estabelecimentos_preparados['flag_inapta'].sum()),
    'enderecos_com_cep': int(estabelecimentos_preparados['cep_limpo'].fillna('').str.len().eq(8).sum()),
    'enderecos_com_numero': int(estabelecimentos_preparados['numero_limpo'].fillna('').ne('').sum())
}

with (SAIDA_PREPARADO / 'qualidade_preparacao.json').open('w', encoding='utf-8') as arquivo:
    json.dump(qualidade_preparacao, arquivo, ensure_ascii=False, indent=2, default=str)

qualidade_preparacao